In [1]:
# Project Review Status Classification - Model Training
# ====================================================

# This notebook handles the model training for the project review status classification.
# It includes model selection, training, evaluation, and interpretation.

# Table of Contents:
# 1. Import Libraries and Load Processed Data
# 2. Feature Selection and Preparation
# 3. Baseline Model Development
# 4. Advanced Model Training
# 5. Model Evaluation and Comparison
# 6. Feature Importance Analysis
# 7. Model Fine-Tuning
# 8. Model Interpretation
# 9. Model Serialization for Deployment

# 1. Import Libraries and Load Processed Data
# -------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_curve, auc
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Set random seed for reproducibility
np.random.seed(42)

# Load the processed data
df = pd.read_csv('processed_project_data.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Review Status Distribution:\n{df['review_status'].value_counts()}")

Dataset Shape: (10000, 42)
Review Status Distribution:
review_status
Approved        4048
Rejected        2994
Needs Review    2958
Name: count, dtype: int64


In [7]:
# 2. Feature Selection and Preparation
# -----------------------------------

# Define target variable
target = 'review_status'
y = df[target]

# Remove the target from features
X = df.drop(target, axis=1)

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Number of numerical features: {len(numerical_features)}")
print(f"Number of categorical features: {len(categorical_features)}")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Testing set shape: {X_test.shape}, {y_test.shape}")

# Create a preprocessor for the pipeline
# - Standard scaling for numerical features
# - One-hot encoding for categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))  # Changed sparse_output to sparse
        ]), categorical_features)
    ]
)

Number of numerical features: 32
Number of categorical features: 9
Training set shape: (8000, 41), (8000,)
Testing set shape: (2000, 41), (2000,)


In [10]:
# 3. Baseline Model Development
# ----------------------------
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

print("Training baseline models...")

# Dummy classifier (majority class strategy)
dummy_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])

# Logistic Regression
lr_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Decision Tree
dt_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Train the baseline models
dummy_clf.fit(X_train, y_train)
lr_clf.fit(X_train, y_train)
dt_clf.fit(X_train, y_train)

# Evaluate baseline models using cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y, model_name="Model"):
    """Evaluate model using cross-validation and return scores"""
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f"{model_name} CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    
    # Make predictions on test set
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    
    print(f"{model_name} Test Accuracy: {accuracy:.4f}")
    print(f"{model_name} Precision: {precision:.4f}")
    print(f"{model_name} Recall: {recall:.4f}")
    print(f"{model_name} F1-Score: {f1:.4f}")
    
    # Create and display confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name.lower().replace(" ", "_")}.png')
    plt.close()
    
    # Display classification report
    print(f"\nClassification Report - {model_name}:")
    print(classification_report(y_test, y_pred))
    
    return {
        'model_name': model_name,
        'cv_accuracy': cv_scores.mean(),
        'test_accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Evaluate baseline models
print("\nEvaluating baseline models...")
baseline_results = []
baseline_results.append(evaluate_model(dummy_clf, X_train, y_train, "Dummy Classifier"))
baseline_results.append(evaluate_model(lr_clf, X_train, y_train, "Logistic Regression"))
baseline_results.append(evaluate_model(dt_clf, X_train, y_train, "Decision Tree"))

# Create comparison dataframe for baseline models
baseline_df = pd.DataFrame(baseline_results)
print("\nBaseline Model Comparison:")
print(baseline_df)

Training baseline models...

Evaluating baseline models...
Dummy Classifier CV Accuracy: 0.4048 (±0.0003)
Dummy Classifier Test Accuracy: 0.4050
Dummy Classifier Precision: 0.1640
Dummy Classifier Recall: 0.4050
Dummy Classifier F1-Score: 0.2335

Classification Report - Dummy Classifier:
              precision    recall  f1-score   support

    Approved       0.41      1.00      0.58       810
Needs Review       0.00      0.00      0.00       591
    Rejected       0.00      0.00      0.00       599

    accuracy                           0.41      2000
   macro avg       0.14      0.33      0.19      2000
weighted avg       0.16      0.41      0.23      2000

Logistic Regression CV Accuracy: 0.9850 (±0.0011)
Logistic Regression Test Accuracy: 0.9915
Logistic Regression Precision: 0.9915
Logistic Regression Recall: 0.9915
Logistic Regression F1-Score: 0.9915

Classification Report - Logistic Regression:
              precision    recall  f1-score   support

    Approved       1.00    

In [16]:
# 4. Advanced Model Training
# -------------------------
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC

# Remove XGBoost since it's not installed
print("\nTraining advanced models...")

# Random Forest
rf_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Gradient Boosting
gb_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

# SVM
svm_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True, random_state=42))
])

# Train the advanced models
rf_clf.fit(X_train, y_train)
gb_clf.fit(X_train, y_train)
svm_clf.fit(X_train, y_train)

# Evaluate advanced models
print("\nEvaluating advanced models...")
advanced_results = []
advanced_results.append(evaluate_model(rf_clf, X_train, y_train, "Random Forest"))
advanced_results.append(evaluate_model(gb_clf, X_train, y_train, "Gradient Boosting"))
advanced_results.append(evaluate_model(svm_clf, X_train, y_train, "SVM"))

# Create comparison dataframe for advanced models
advanced_df = pd.DataFrame(advanced_results)
print("\nAdvanced Model Comparison:")
print(advanced_df)

# Combine all results
all_results = pd.concat([baseline_df, advanced_df], ignore_index=True)
all_results = all_results.sort_values('f1', ascending=False)

# Plot model comparison
plt.figure(figsize=(12, 8))
sns.barplot(x='model_name', y='f1', data=all_results)
plt.title('Model Comparison - F1 Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('model_comparison_f1.png')
plt.close()


Training advanced models...

Evaluating advanced models...
Random Forest CV Accuracy: 0.9892 (±0.0009)
Random Forest Test Accuracy: 0.9945
Random Forest Precision: 0.9945
Random Forest Recall: 0.9945
Random Forest F1-Score: 0.9945

Classification Report - Random Forest:
              precision    recall  f1-score   support

    Approved       1.00      1.00      1.00       810
Needs Review       1.00      0.98      0.99       591
    Rejected       0.99      1.00      1.00       599

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000

Gradient Boosting CV Accuracy: 0.9914 (±0.0012)
Gradient Boosting Test Accuracy: 0.9930
Gradient Boosting Precision: 0.9930
Gradient Boosting Recall: 0.9930
Gradient Boosting F1-Score: 0.9930

Classification Report - Gradient Boosting:
              precision    recall  f1-score   support

    Approved       1.00      1.00      1.00       810
Ne

In [19]:
# 5. Model Evaluation and Comparison
# ---------------------------------

# Identify the best model
best_model_name = all_results.iloc[0]['model_name']
print(f"\nBest performing model: {best_model_name}")

# Select the best model
if best_model_name == "Random Forest":
    best_model = rf_clf
elif best_model_name == "Gradient Boosting":
    best_model = gb_clf
elif best_model_name == "XGBoost":
    best_model = xgb_clf
elif best_model_name == "SVM":
    best_model = svm_clf
elif best_model_name == "Logistic Regression":
    best_model = lr_clf
elif best_model_name == "Decision Tree":
    best_model = dt_clf
else:
    best_model = dummy_clf

# Perform more detailed evaluation on the best model
# ROC curves for multiclass
def plot_roc_curves(model, X, y, model_name):
    """Plot ROC curves for multiclass classification"""
    y_pred_proba = model.predict_proba(X)
    
    # Get the unique classes
    classes = model.classes_
    n_classes = len(classes)
    
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    y_test_bin = pd.get_dummies(y).values
    
    plt.figure(figsize=(10, 8))
    
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
        plt.plot(fpr[i], tpr[i], lw=2,
                 label=f'ROC curve (class {cls}, area = {roc_auc[i]:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves - {model_name}')
    plt.legend(loc="lower right")
    plt.savefig(f'roc_curves_{model_name.lower().replace(" ", "_")}.png')
    plt.close()

# Plot ROC curves for the best model
plot_roc_curves(best_model, X_test, y_test, best_model_name)


Best performing model: Random Forest


In [22]:
# 6. Feature Importance Analysis
# -----------------------------

def analyze_feature_importance(model, feature_names, model_name):
    """Extract and visualize feature importances from the model"""
    # Check if the model has feature_importances_ attribute
    if hasattr(model[-1], 'feature_importances_'):
        importances = model[-1].feature_importances_
        
        # Get preprocessed feature names
        if hasattr(model[0], 'get_feature_names_out'):
            preprocessed_features = model[0].get_feature_names_out()
        else:
            # For older scikit-learn versions
            num_features = [f"num__{f}" for f in numerical_features]
            
            # Get categorical feature names after one-hot encoding
            cat_encoder = model[0].transformers_[1][1].named_steps['onehot']
            cat_features = cat_encoder.get_feature_names_out(categorical_features)
            cat_features = [f"cat__{f}" for f in cat_features]
            
            preprocessed_features = np.concatenate([num_features, cat_features])
        
        # Create dataframe of feature importances
        feature_importance = pd.DataFrame({
            'feature': preprocessed_features,
            'importance': importances
        })
        
        # Sort by importance
        feature_importance = feature_importance.sort_values('importance', ascending=False)
        
        # Plot top 20 features
        plt.figure(figsize=(12, 10))
        top_features = feature_importance.head(20)
        sns.barplot(x='importance', y='feature', data=top_features)
        plt.title(f'Top 20 Feature Importances - {model_name}')
        plt.tight_layout()
        plt.savefig(f'feature_importance_{model_name.lower().replace(" ", "_")}.png')
        plt.close()
        
        return feature_importance
    elif hasattr(model[-1], 'coef_'):
        # For linear models like Logistic Regression
        coefficients = model[-1].coef_
        
        if len(coefficients.shape) > 1:
            # Multiclass case - average the absolute coefficients across classes
            importances = np.mean(np.abs(coefficients), axis=0)
        else:
            importances = np.abs(coefficients)
        
        # Get preprocessed feature names
        if hasattr(model[0], 'get_feature_names_out'):
            preprocessed_features = model[0].get_feature_names_out()
        else:
            # For older scikit-learn versions
            num_features = [f"num__{f}" for f in numerical_features]
            
            # Get categorical feature names after one-hot encoding
            cat_encoder = model[0].transformers_[1][1].named_steps['onehot']
            cat_features = cat_encoder.get_feature_names_out(categorical_features)
            cat_features = [f"cat__{f}" for f in cat_features]
            
            preprocessed_features = np.concatenate([num_features, cat_features])
        
        # Create dataframe of feature importances
        feature_importance = pd.DataFrame({
            'feature': preprocessed_features,
            'importance': importances
        })
        
        # Sort by importance
        feature_importance = feature_importance.sort_values('importance', ascending=False)
        
        # Plot top 20 features
        plt.figure(figsize=(12, 10))
        top_features = feature_importance.head(20)
        sns.barplot(x='importance', y='feature', data=top_features)
        plt.title(f'Top 20 Feature Importances - {model_name}')
        plt.tight_layout()
        plt.savefig(f'feature_importance_{model_name.lower().replace(" ", "_")}.png')
        plt.close()
        
        return feature_importance
    else:
        print(f"Feature importance not available for {model_name}")
        return None

# Analyze feature importance for tree-based models
if best_model_name in ["Random Forest", "Gradient Boosting", "XGBoost", "Decision Tree"]:
    feature_importance = analyze_feature_importance(best_model, X.columns, best_model_name)
    
    if feature_importance is not None:
        print("\nTop 10 Important Features:")
        print(feature_importance.head(10))



Top 10 Important Features:
                                feature  importance
0   num__synopsis_plagiarism_similarity    0.720060
28   num__casting_description_sentiment    0.014234
11            num__synopsis_readability    0.013319
10         num__description_readability    0.013150
8            num__description_sentiment    0.012549
29           num__project_duration_days    0.012355
2               num__description_length    0.012246
9               num__synopsis_sentiment    0.011959
3                  num__synopsis_length    0.011712
24        num__avg_casting_compensation    0.010956


In [25]:
# 7. Model Fine-Tuning
# -------------------

def fine_tune_model(model, X, y, param_grid, model_name):
    """Fine-tune model using grid search"""
    print(f"\nFine-tuning {model_name}...")
    
    # Create grid search
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1
    )
    
    # Fit grid search
    grid_search.fit(X, y)
    
    # Best parameters and score
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")
    
    # Evaluate best model
    best_model = grid_search.best_estimator_
    result = evaluate_model(best_model, X, y, f"{model_name} (Tuned)")
    
    return best_model, result

# Define parameter grids for fine-tuning based on the best model
if best_model_name == "Random Forest":
    param_grid = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5, 10]
    }
    tuned_model, tuned_result = fine_tune_model(rf_clf, X_train, y_train, param_grid, "Random Forest")

elif best_model_name == "Gradient Boosting":
    param_grid = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
        'classifier__max_depth': [3, 5, 7]
    }
    tuned_model, tuned_result = fine_tune_model(gb_clf, X_train, y_train, param_grid, "Gradient Boosting")

elif best_model_name == "XGBoost":
    param_grid = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
        'classifier__max_depth': [3, 5, 7]
    }
    tuned_model, tuned_result = fine_tune_model(xgb_clf, X_train, y_train, param_grid, "XGBoost")

elif best_model_name == "SVM":
    param_grid = {
        'classifier__C': [0.1, 1, 10],
        'classifier__gamma': ['scale', 'auto', 0.1, 1],
        'classifier__kernel': ['rbf', 'linear']
    }
    tuned_model, tuned_result = fine_tune_model(svm_clf, X_train, y_train, param_grid, "SVM")

elif best_model_name == "Logistic Regression":
    param_grid = {
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2', 'elasticnet', None],
        'classifier__solver': ['newton-cg', 'lbfgs', 'liblinear']
    }
    tuned_model, tuned_result = fine_tune_model(lr_clf, X_train, y_train, param_grid, "Logistic Regression")

elif best_model_name == "Decision Tree":
    param_grid = {
        'classifier__max_depth': [None, 5, 10, 15],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [1, 2, 4]
    }
    tuned_model, tuned_result = fine_tune_model(dt_clf, X_train, y_train, param_grid, "Decision Tree")

else:
    print("Skipping fine-tuning for Dummy Classifier")
    tuned_model = best_model
    tuned_result = None

# Compare original vs. tuned model if applicable
if tuned_result is not None:
    comparison = pd.DataFrame([
        {'model': best_model_name, 'accuracy': all_results.iloc[0]['test_accuracy'], 'f1': all_results.iloc[0]['f1']},
        {'model': f"{best_model_name} (Tuned)", 'accuracy': tuned_result['test_accuracy'], 'f1': tuned_result['f1']}
    ])
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='model', y='f1', data=comparison)
    plt.title('Performance Comparison: Original vs. Tuned Model')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig('tuned_model_comparison.png')
    plt.close()
    
    # Update best model
    final_model = tuned_model
else:
    final_model = best_model



Fine-tuning Random Forest...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 50}
Best CV score: 0.9911
Random Forest (Tuned) CV Accuracy: 0.9911 (±0.0011)
Random Forest (Tuned) Test Accuracy: 0.9945
Random Forest (Tuned) Precision: 0.9945
Random Forest (Tuned) Recall: 0.9945
Random Forest (Tuned) F1-Score: 0.9945

Classification Report - Random Forest (Tuned):
              precision    recall  f1-score   support

    Approved       1.00      1.00      1.00       810
Needs Review       0.99      0.99      0.99       591
    Rejected       0.99      1.00      1.00       599

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000



In [28]:
# 8. Model Interpretation
# ----------------------
from sklearn.inspection import permutation_importance

# Permutation feature importance
def permutation_feature_importance(model, X, y, model_name):
    """Calculate permutation feature importance"""
    print(f"\nCalculating permutation feature importance for {model_name}...")
    
    # Perform permutation importance
    result = permutation_importance(model, X, y, n_repeats=5, random_state=42, n_jobs=-1)
    
    # Get feature names
    if hasattr(model[0], 'get_feature_names_out'):
        feature_names = model[0].get_feature_names_out()
    else:
        # For older scikit-learn versions
        num_features = [f"num__{f}" for f in numerical_features]
        
        # Get categorical feature names after one-hot encoding
        cat_encoder = model[0].transformers_[1][1].named_steps['onehot']
        cat_features = cat_encoder.get_feature_names_out(categorical_features)
        cat_features = [f"cat__{f}" for f in cat_features]
        
        feature_names = np.concatenate([num_features, cat_features])
    
    # Create dataframe
    perm_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': result.importances_mean,
        'std': result.importances_std
    })
    
    # Sort by importance
    perm_importance = perm_importance.sort_values('importance', ascending=False)
    
    # Plot top 20 features
    plt.figure(figsize=(12, 10))
    top_features = perm_importance.head(20)
    sns.barplot(x='importance', y='feature', data=top_features)
    plt.title(f'Top 20 Permutation Feature Importances - {model_name}')
    plt.tight_layout()
    plt.savefig(f'permutation_importance_{model_name.lower().replace(" ", "_")}.png')
    plt.close()
    
    return perm_importance

# Calculate permutation importance for the final model
try:
    perm_importance = permutation_feature_importance(final_model, X_test, y_test, 
                                                  best_model_name + " (Final)")
    print("\nTop 10 Features (Permutation Importance):")
    print(perm_importance.head(10))
except Exception as e:
    print(f"Error calculating permutation importance: {e}")


Calculating permutation feature importance for Random Forest (Final)...
Error calculating permutation importance: All arrays must be of the same length


In [31]:
# 9. Model Serialization for Deployment
# -----------------------------------

# Save the final model
print("\nSaving the final model...")
model_filename = f'project_review_classifier_{best_model_name.lower().replace(" ", "_")}.pkl'
joblib.dump(final_model, model_filename)
print(f"Model saved as '{model_filename}'")

# Create a simple prediction function
def predict_review_status(model, project_data):
    """
    Make predictions on new project data
    
    Args:
        model: Trained pipeline model
        project_data: DataFrame with project features
    
    Returns:
        Predicted review status and probabilities
    """
    # Ensure project_data has all required columns
    required_cols = list(X.columns)
    missing_cols = set(required_cols) - set(project_data.columns)
    
    if missing_cols:
        print(f"Warning: Missing columns in input data: {missing_cols}")
        for col in missing_cols:
            project_data[col] = np.nan
    
    # Select only the columns used during training
    project_data = project_data[required_cols]
    
    # Make predictions
    prediction = model.predict(project_data)[0]
    probabilities = model.predict_proba(project_data)[0]
    
    # Create probability dictionary
    prob_dict = {cls: prob for cls, prob in zip(model.classes_, probabilities)}
    
    return {
        'prediction': prediction,
        'probabilities': prob_dict
    }

# Example of using the prediction function
print("\nExample of using the prediction function:")
# Use the first record from the test set as an example
example_project = X_test.iloc[[0]].copy()
prediction_result = predict_review_status(final_model, example_project)
print(f"Predicted review status: {prediction_result['prediction']}")
print(f"Prediction probabilities: {prediction_result['probabilities']}")

# Save example code for making predictions
example_code = """
import pandas as pd
import joblib

# Load the model
model = joblib.load('project_review_classifier_model.pkl')

# Prepare project data (ensure it has all required features)
project_data = pd.DataFrame({
    # Include all project features here
    'project_type_id': ['Documentary'],
    'title': ['Sample Project Title'],
    'description': ['This is a sample project description.'],
    # ... other features
})

# Make prediction
prediction = model.predict(project_data)[0]
probabilities = model.predict_proba(project_data)[0]

print(f"Predicted review status: {prediction}")
print(f"Prediction probabilities: {dict(zip(model.classes_, probabilities))}")
"""

with open('example_usage.py', 'w') as f:
    f.write(example_code)

print("\nExample code saved as 'example_usage.py'")

# Summary of results
print("\nModel Training Summary:")
print(f"Best model: {best_model_name}")
print(f"Test accuracy: {all_results.iloc[0]['test_accuracy']:.4f}")
print(f"F1 score: {all_results.iloc[0]['f1']:.4f}")
print("\nModel saved and ready for deployment!")

# Final visualization - decision boundaries (if feasible)
# For high-dimensional data, we can use t-SNE to visualize in 2D
from sklearn.manifold import TSNE

# Only do this if the dataset is not too large
if X.shape[0] < 1000:
    print("\nCreating t-SNE visualization of decision boundaries...")
    
    # Get preprocessed data
    X_preprocessed = final_model[0].transform(X)
    
    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X_preprocessed)
    
    # Create DataFrame for plotting
    tsne_df = pd.DataFrame({
        'x': X_tsne[:, 0],
        'y': X_tsne[:, 1],
        'review_status': y
    })
    
    # Plot
    plt.figure(figsize=(12, 10))
    sns.scatterplot(x='x', y='y', hue='review_status', data=tsne_df, palette='viridis')
    plt.title('t-SNE Visualization of Project Data with Review Status')
    plt.savefig('tsne_visualization.png')
    plt.close()


Saving the final model...
Model saved as 'project_review_classifier_random_forest.pkl'

Example of using the prediction function:
Predicted review status: Needs Review
Prediction probabilities: {'Approved': 0.11242223098611187, 'Needs Review': 0.6386678595221134, 'Rejected': 0.24890990949177458}

Example code saved as 'example_usage.py'

Model Training Summary:
Best model: Random Forest
Test accuracy: 0.9945
F1 score: 0.9945

Model saved and ready for deployment!
